In [39]:
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
import numpy as np
import random
import pandas as pd
import matplotlib.pyplot as plt
import os
import copy
import seaborn as sns

from sklearn import preprocessing
from sklearn.metrics import log_loss
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import warnings
warnings.filterwarnings('ignore')

In [40]:
from sklearn.preprocessing import QuantileTransformer

In [41]:
train_features = pd.read_csv('train_features.csv')
train_targets_scored = pd.read_csv('train_targets_scored.csv')
train_targets_nonscored = pd.read_csv('train_targets_nonscored.csv')

test_features = pd.read_csv('test_features.csv')
sample_submission = pd.read_csv('sample_submission.csv')

In [42]:
GENES = [col for col in train_features.columns if col.startswith('g-')]
CELLS = [col for col in train_features.columns if col.startswith('c-')]

In [43]:
#RankGauss

for col in (GENES + CELLS):

    transformer = QuantileTransformer(n_quantiles=100,random_state=0, output_distribution="normal")
    vec_len = len(train_features[col].values)
    vec_len_test = len(test_features[col].values)
    raw_vec = train_features[col].values.reshape(vec_len, 1)
    transformer.fit(raw_vec)

    train_features[col] = transformer.transform(raw_vec).reshape(1, vec_len)[0]
    test_features[col] = transformer.transform(test_features[col].values.reshape(vec_len_test, 1)).reshape(1, vec_len_test)[0]

In [44]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    
seed_everything(seed=42)

In [45]:
# GENES
n_comp = xx  #<--Update 

data = pd.concat([pd.DataFrame(train_features[GENES]), pd.DataFrame(test_features[GENES])])
data2 = (PCA(n_components=n_comp, random_state=42).fit_transform(data[GENES]))
train2 = data2[:train_features.shape[0]]; test2 = data2[-test_features.shape[0]:]

train2 = pd.DataFrame(train2, columns=[f'pca_G-{i}' for i in range(n_comp)])
test2 = pd.DataFrame(test2, columns=[f'pca_G-{i}' for i in range(n_comp)])

# drop_cols = [f'c-{i}' for i in range(n_comp,len(GENES))]
train_features = pd.concat((train_features, train2), axis=1)
test_features = pd.concat((test_features, test2), axis=1)

In [46]:
#CELLS
n_comp = xx  #<--Update 

data = pd.concat([pd.DataFrame(train_features[CELLS]), pd.DataFrame(test_features[CELLS])])
data2 = (PCA(n_components=n_comp, random_state=42).fit_transform(data[CELLS]))
train2 = data2[:train_features.shape[0]]; test2 = data2[-test_features.shape[0]:]

train2 = pd.DataFrame(train2, columns=[f'pca_C-{i}' for i in range(n_comp)])
test2 = pd.DataFrame(test2, columns=[f'pca_C-{i}' for i in range(n_comp)])

# drop_cols = [f'c-{i}' for i in range(n_comp,len(CELLS))]
train_features = pd.concat((train_features, train2), axis=1)
test_features = pd.concat((test_features, test2), axis=1)

In [47]:
train_features.shape

(23814, 1526)

In [48]:
from sklearn.feature_selection import VarianceThreshold


var_thresh = VarianceThreshold(xx)  #<-- Update
data = train_features.append(test_features)
data_transformed = var_thresh.fit_transform(data.iloc[:, 4:])

train_features_transformed = data_transformed[ : train_features.shape[0]]
test_features_transformed = data_transformed[-test_features.shape[0] : ]


train_features = pd.DataFrame(train_features[['sig_id','cp_type','cp_time','cp_dose']].values.reshape(-1, 4),\
                              columns=['sig_id','cp_type','cp_time','cp_dose'])

train_features = pd.concat([train_features, pd.DataFrame(train_features_transformed)], axis=1)


test_features = pd.DataFrame(test_features[['sig_id','cp_type','cp_time','cp_dose']].values.reshape(-1, 4),\
                             columns=['sig_id','cp_type','cp_time','cp_dose'])

test_features = pd.concat([test_features, pd.DataFrame(test_features_transformed)], axis=1)

train_features.shape

(23814, 1040)

In [49]:
from sklearn.cluster import KMeans
def fe_cluster(train, test, n_clusters_g = xx, n_clusters_c = xx, SEED = 123): # try different values
    
    features_g = list(train.columns[4:776])
    features_c = list(train.columns[776:876])
    
    def create_cluster(train, test, features, kind = 'g', n_clusters = n_clusters_g):
        train_ = train[features].copy()
        test_ = test[features].copy()
        data = pd.concat([train_, test_], axis = 0)
        kmeans = KMeans(n_clusters = n_clusters, random_state = SEED).fit(data)
        train[f'clusters_{kind}'] = kmeans.labels_[:train.shape[0]]
        test[f'clusters_{kind}'] = kmeans.labels_[train.shape[0]:]
        train = pd.get_dummies(train, columns = [f'clusters_{kind}'])
        test = pd.get_dummies(test, columns = [f'clusters_{kind}'])
        return train, test
    
    train, test = create_cluster(train, test, features_g, kind = 'g', n_clusters = n_clusters_g)
    train, test = create_cluster(train, test, features_c, kind = 'c', n_clusters = n_clusters_c)
    return train, test

train_features ,test_features=fe_cluster(train_features,test_features)
train_features.shape

(23814, 1080)

In [50]:
def fe_stats(train, test):
    
    features_g = list(train.columns[4:776])
    features_c = list(train.columns[776:876])
    
    for df in train, test:
        df['g_sum'] = df[features_g].sum(axis = 1)
        df['g_mean'] = df[features_g].mean(axis = 1)
        df['g_std'] = df[features_g].std(axis = 1)
        df['g_kurt'] = df[features_g].kurtosis(axis = 1) # whether it is normal distribution
        df['g_skew'] = df[features_g].skew(axis = 1) # whether it is normal distribution
        df['c_sum'] = df[features_c].sum(axis = 1)
        df['c_mean'] = df[features_c].mean(axis = 1)
        df['c_std'] = df[features_c].std(axis = 1)
        df['c_kurt'] = df[features_c].kurtosis(axis = 1)
        df['c_skew'] = df[features_c].skew(axis = 1)
        df['gc_sum'] = df[features_g + features_c].sum(axis = 1)
        df['gc_mean'] = df[features_g + features_c].mean(axis = 1)
        df['gc_std'] = df[features_g + features_c].std(axis = 1)
        df['gc_kurt'] = df[features_g + features_c].kurtosis(axis = 1)
        df['gc_skew'] = df[features_g + features_c].skew(axis = 1)
        
    return train, test

train_features,test_features=fe_stats(train_features,test_features)
train_features.shape

(21948, 1245)

In [51]:
train = train_features.merge(train_targets_scored, on='sig_id')
train = train[train['cp_type']!='ctl_vehicle'].reset_index(drop=True)
test = test_features[test_features['cp_type']!='ctl_vehicle'].reset_index(drop=True)

target = train[train_targets_scored.columns]

In [52]:
train = train.drop('cp_type', axis=1)
test = test.drop('cp_type', axis=1)

In [53]:
target_cols = target.drop('sig_id', axis=1).columns.values.tolist()

In [54]:
folds = train.copy()

mskf = MultilabelStratifiedKFold(n_splits=xx)

for f, (t_idx, v_idx) in enumerate(mskf.split(X=train, y=target)):
    folds.loc[v_idx, 'kfold'] = int(f)

folds['kfold'] = folds['kfold'].astype(int)

In [55]:
print(train.shape)
print(folds.shape)
print(test.shape)
print(target.shape)
print(sample_submission.shape)

(21948, 1300)
(21948, 1301)
(3624, 1094)
(21948, 207)
(3982, 207)


In [56]:
class MoADataset:
    def __init__(self, features, targets):
        self.features = features
        self.targets = targets
        
    def __len__(self):
        return (self.features.shape[0])
    
    def __getitem__(self, idx):
        dct = {
            'x' : torch.tensor(self.features[idx, :], dtype=torch.float),
            'y' : torch.tensor(self.targets[idx, :], dtype=torch.float)            
        }
        return dct
    
class TestDataset:
    def __init__(self, features):
        self.features = features
        
    def __len__(self):
        return (self.features.shape[0])
    
    def __getitem__(self, idx):
        dct = {
            'x' : torch.tensor(self.features[idx, :], dtype=torch.float)
        }
        return dct

In [57]:
def train_fn(model, optimizer, scheduler, loss_fn, dataloader, device):
    model.train()
    final_loss = 0
    
    for data in dataloader:
        optimizer.zero_grad()
        inputs, targets = data['x'].to(device), data['y'].to(device)
#         print(inputs.shape)
        outputs = model(inputs)
        loss = loss_fn(outputs, targets)
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        final_loss += loss.item()
        
    final_loss /= len(dataloader)
    
    return final_loss


def valid_fn(model, loss_fn, dataloader, device):
    model.eval()
    final_loss = 0
    valid_preds = []
    
    for data in dataloader:
        inputs, targets = data['x'].to(device), data['y'].to(device)
        outputs = model(inputs)
        loss = loss_fn(outputs, targets)
        
        final_loss += loss.item()
        valid_preds.append(outputs.sigmoid().detach().cpu().numpy())
        
    final_loss /= len(dataloader)
    valid_preds = np.concatenate(valid_preds)
    
    return final_loss, valid_preds

def inference_fn(model, dataloader, device):
    model.eval()
    preds = []
    
    for data in dataloader:
        inputs = data['x'].to(device)

        with torch.no_grad():
            outputs = model(inputs)
        
        preds.append(outputs.sigmoid().detach().cpu().numpy())
        
    preds = np.concatenate(preds)
    
    return preds

In [58]:
import torch
from torch.nn.modules.loss import _WeightedLoss
import torch.nn.functional as F

class SmoothBCEwLogits(_WeightedLoss):
    def __init__(self, weight=None, reduction='mean', smoothing=0.0):
        super().__init__(weight=weight, reduction=reduction)
        self.smoothing = smoothing
        self.weight = weight
        self.reduction = reduction

    @staticmethod
    def _smooth(targets:torch.Tensor, n_labels:int, smoothing=0.0):
        assert 0 <= smoothing < 1
        with torch.no_grad():
            targets = targets * (1.0 - smoothing) + 0.5 * smoothing
        return targets

    def forward(self, inputs, targets):
        targets = SmoothBCEwLogits._smooth(targets, inputs.size(-1),
            self.smoothing)
        loss = F.binary_cross_entropy_with_logits(inputs, targets,self.weight)

        if  self.reduction == 'sum':
            loss = loss.sum()
        elif  self.reduction == 'mean':
            loss = loss.mean()

        return loss

In [59]:
class Model(nn.Module):
    def __init__(self, num_features, num_targets, hidden_size):
        super(Model, self).__init__()
        self.batch_norm1 = nn.BatchNorm1d(num_features)
        self.dense1 = nn.utils.weight_norm(nn.Linear(num_features, hidden_size))
        
        self.batch_norm2 = nn.BatchNorm1d(hidden_size)
        self.dropout2 = nn.Dropout(0.2) # update, try value between 0.2-0.4
        self.dense2 = nn.utils.weight_norm(nn.Linear(hidden_size, hidden_size))
        
        self.batch_norm3 = nn.BatchNorm1d(hidden_size)
        self.dropout3 = nn.Dropout(0.2) #update
        self.dense3 = nn.utils.weight_norm(nn.Linear(hidden_size, num_targets))
    
    def forward(self, x):
        x = self.batch_norm1(x)
        x = F.leaky_relu(self.dense1(x))#update
        #https://www.geeksforgeeks.org/python-tensorflow-nn-relu-and-nn-leaky_relu/
        
        x = self.batch_norm2(x)
        x = self.dropout2(x)
        x = F.leaky_relu(self.dense2(x))#update
        
        x = self.batch_norm3(x)
        x = self.dropout3(x)
        x = self.dense3(x)
        
        return x
    
class LabelSmoothingLoss(nn.Module):
    def __init__(self, classes, smoothing=0.0, dim=-1):
        super(LabelSmoothingLoss, self).__init__()
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing
        self.cls = classes
        self.dim = dim

    def forward(self, pred, target):
        pred = pred.log_softmax(dim=self.dim)
        with torch.no_grad():
            # true_dist = pred.data.clone()
            true_dist = torch.zeros_like(pred)
            true_dist.fill_(self.smoothing / (self.cls - 1))
            true_dist.scatter_(1, target.data.unsqueeze(1), self.confidence)
        return torch.mean(torch.sum(-true_dist * pred, dim=self.dim))

In [60]:
def process_data(data):
    data = pd.get_dummies(data, columns=['cp_time','cp_dose'])
    return data

In [61]:
feature_cols = [c for c in process_data(folds).columns if c not in target_cols]
feature_cols = [c for c in feature_cols if c not in ['kfold','sig_id']]
len(feature_cols)

1096

In [62]:
# HyperParameters

DEVICE = ('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS = xx
BATCH_SIZE = xx # 64 (2^6), 128 (2^7), 256 (2^8), 512
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
NFOLDS = xx           
EARLY_STOPPING_STEPS = 10
EARLY_STOP = False

num_features=len(feature_cols)
num_targets=len(target_cols)
hidden_size=1500  #reduce hidden size, 1000-1500, update

In [63]:
def run_training(fold, seed):
    
    seed_everything(seed)
    
    train = process_data(folds)
    test_ = process_data(test)
    
    trn_idx = train[train['kfold'] != fold].index
    val_idx = train[train['kfold'] == fold].index
    
    train_df = train[train['kfold'] != fold].reset_index(drop=True)
    valid_df = train[train['kfold'] == fold].reset_index(drop=True)
    
    x_train, y_train  = train_df[feature_cols].values, train_df[target_cols].values
    x_valid, y_valid =  valid_df[feature_cols].values, valid_df[target_cols].values
    
    train_dataset = MoADataset(x_train, y_train)
    valid_dataset = MoADataset(x_valid, y_valid)
    trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    validloader = torch.utils.data.DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    model = Model(
        num_features=num_features,
        num_targets=num_targets,
        hidden_size=hidden_size,
    )
    
    model.to(DEVICE)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.OneCycleLR(optimizer=optimizer, pct_start=0.1, div_factor=1e3, 
                                              max_lr=1e-2, epochs=EPOCHS, steps_per_epoch=len(trainloader))
    
    loss_fn = nn.BCEWithLogitsLoss()
    loss_tr = SmoothBCEwLogits(smoothing =0.001)
    
    early_stopping_steps = EARLY_STOPPING_STEPS
    early_step = 0
   
    oof = np.zeros((len(train), target.iloc[:, 1:].shape[1]))
    best_loss = np.inf
    
    for epoch in range(EPOCHS):
        
        train_loss = train_fn(model, optimizer,scheduler, loss_tr, trainloader, DEVICE)
        print(f"FOLD: {fold}, EPOCH: {epoch}, train_loss: {train_loss}")
        valid_loss, valid_preds = valid_fn(model, loss_fn, validloader, DEVICE)
        print(f"FOLD: {fold}, EPOCH: {epoch}, valid_loss: {valid_loss}")
        
        if valid_loss < best_loss:
            
            best_loss = valid_loss
            oof[val_idx] = valid_preds
            torch.save(model.state_dict(), f"FOLD{fold}_.pth")
        
        elif(EARLY_STOP == True):
            
            early_step += 1
            if (early_step >= early_stopping_steps):
                break
            
    
    #--------------------- PREDICTION---------------------
    x_test = test_[feature_cols].values
    testdataset = TestDataset(x_test)
    testloader = torch.utils.data.DataLoader(testdataset, batch_size=BATCH_SIZE, shuffle=False)
    
    model = Model(
        num_features=num_features,
        num_targets=num_targets,
        hidden_size=hidden_size,

    )
    
    model.load_state_dict(torch.load(f"FOLD{fold}_.pth"))
    model.to(DEVICE)
    
    predictions = np.zeros((len(test_), target.iloc[:, 1:].shape[1]))
    predictions = inference_fn(model, testloader, DEVICE)
    
    return oof, predictions

In [38]:
def run_k_fold(NFOLDS, seed):
    oof = np.zeros((len(train), len(target_cols)))
    predictions = np.zeros((len(test), len(target_cols)))
    
    for fold in range(NFOLDS):
        oof_, pred_ = run_training(fold, seed)
        
        predictions += pred_ / NFOLDS
        oof += oof_
        
    return oof, predictions

In [64]:
# Averaging on multiple SEEDS

SEED = [x,x,x,x,x,x,x] #<-- Update
oof = np.zeros((len(train), len(target_cols)))
predictions = np.zeros((len(test), len(target_cols)))

for seed in SEED:
    
    oof_, predictions_ = run_k_fold(NFOLDS, seed)
    oof += oof_ / len(SEED)
    predictions += predictions_ / len(SEED)

train[target_cols] = oof

FOLD: 0, EPOCH: 0, train_loss: 0.46670162736051746
FOLD: 0, EPOCH: 0, valid_loss: 0.024329666942358018
FOLD: 0, EPOCH: 1, train_loss: 0.023630513372469922
FOLD: 0, EPOCH: 1, valid_loss: 0.019039385691285132
FOLD: 0, EPOCH: 2, train_loss: 0.021768843793139165
FOLD: 0, EPOCH: 2, valid_loss: 0.018008211627602577
FOLD: 0, EPOCH: 3, train_loss: 0.020611382361982955
FOLD: 0, EPOCH: 3, valid_loss: 0.01736496400088072
FOLD: 0, EPOCH: 4, train_loss: 0.02040786145343667
FOLD: 0, EPOCH: 4, valid_loss: 0.017421888001263142
FOLD: 0, EPOCH: 5, train_loss: 0.02035410344904783
FOLD: 0, EPOCH: 5, valid_loss: 0.01761452779173851
FOLD: 0, EPOCH: 6, train_loss: 0.020377318507858684
FOLD: 0, EPOCH: 6, valid_loss: 0.017546211928129198
FOLD: 0, EPOCH: 7, train_loss: 0.020372414099825483
FOLD: 0, EPOCH: 7, valid_loss: 0.017870953641831874
FOLD: 0, EPOCH: 8, train_loss: 0.020390140032078945
FOLD: 0, EPOCH: 8, valid_loss: 0.017459892295300962
FOLD: 0, EPOCH: 9, train_loss: 0.020292947702363236
FOLD: 0, EPOCH: 9

FOLD: 3, EPOCH: 3, valid_loss: 0.01772825948894024
FOLD: 3, EPOCH: 4, train_loss: 0.020379347093149917
FOLD: 3, EPOCH: 4, valid_loss: 0.017597601935267448
FOLD: 3, EPOCH: 5, train_loss: 0.020344469822993896
FOLD: 3, EPOCH: 5, valid_loss: 0.017658995091915132
FOLD: 3, EPOCH: 6, train_loss: 0.020330298844040657
FOLD: 3, EPOCH: 6, valid_loss: 0.017897670455276966
FOLD: 3, EPOCH: 7, train_loss: 0.020328338555738228
FOLD: 3, EPOCH: 7, valid_loss: 0.017743728682398797
FOLD: 3, EPOCH: 8, train_loss: 0.02038690778521859
FOLD: 3, EPOCH: 8, valid_loss: 0.01743676722049713
FOLD: 3, EPOCH: 9, train_loss: 0.020309540449457916
FOLD: 3, EPOCH: 9, valid_loss: 0.017475240863859653
FOLD: 3, EPOCH: 10, train_loss: 0.020304455656279512
FOLD: 3, EPOCH: 10, valid_loss: 0.01747080996632576
FOLD: 3, EPOCH: 11, train_loss: 0.020264136041102766
FOLD: 3, EPOCH: 11, valid_loss: 0.017324863560497762
FOLD: 3, EPOCH: 12, train_loss: 0.020173654467070184
FOLD: 3, EPOCH: 12, valid_loss: 0.01726184070110321
FOLD: 3, EP

FOLD: 6, EPOCH: 7, train_loss: 0.02038368580191314
FOLD: 6, EPOCH: 7, valid_loss: 0.017295700535178184
FOLD: 6, EPOCH: 8, train_loss: 0.020437574267488757
FOLD: 6, EPOCH: 8, valid_loss: 0.01721000175923109
FOLD: 6, EPOCH: 9, train_loss: 0.020353042058089153
FOLD: 6, EPOCH: 9, valid_loss: 0.017462434619665144
FOLD: 6, EPOCH: 10, train_loss: 0.020406723808066376
FOLD: 6, EPOCH: 10, valid_loss: 0.017282859943807124
FOLD: 6, EPOCH: 11, train_loss: 0.02031809000336394
FOLD: 6, EPOCH: 11, valid_loss: 0.01723184660077095
FOLD: 6, EPOCH: 12, train_loss: 0.020280789913368874
FOLD: 6, EPOCH: 12, valid_loss: 0.017019969560205937
FOLD: 6, EPOCH: 13, train_loss: 0.020061974648107477
FOLD: 6, EPOCH: 13, valid_loss: 0.016955741867423057
FOLD: 6, EPOCH: 14, train_loss: 0.01995629077257753
FOLD: 6, EPOCH: 14, valid_loss: 0.01693059179931879
FOLD: 6, EPOCH: 15, train_loss: 0.019782595222397725
FOLD: 6, EPOCH: 15, valid_loss: 0.01687681593000889
FOLD: 6, EPOCH: 16, train_loss: 0.01960020265256872
FOLD: 6

FOLD: 2, EPOCH: 11, train_loss: 0.020286529800113366
FOLD: 2, EPOCH: 11, valid_loss: 0.017157148346304895
FOLD: 2, EPOCH: 12, train_loss: 0.020257217366070973
FOLD: 2, EPOCH: 12, valid_loss: 0.017133724950253962
FOLD: 2, EPOCH: 13, train_loss: 0.020105723114240737
FOLD: 2, EPOCH: 13, valid_loss: 0.01695586647838354
FOLD: 2, EPOCH: 14, train_loss: 0.019981340473505103
FOLD: 2, EPOCH: 14, valid_loss: 0.016831899397075176
FOLD: 2, EPOCH: 15, train_loss: 0.019786169283649548
FOLD: 2, EPOCH: 15, valid_loss: 0.01692714784294367
FOLD: 2, EPOCH: 16, train_loss: 0.019600779958525483
FOLD: 2, EPOCH: 16, valid_loss: 0.01670025646686554
FOLD: 2, EPOCH: 17, train_loss: 0.019352599801052184
FOLD: 2, EPOCH: 17, valid_loss: 0.016625601649284363
FOLD: 2, EPOCH: 18, train_loss: 0.01904801114582691
FOLD: 2, EPOCH: 18, valid_loss: 0.01644919317215681
FOLD: 2, EPOCH: 19, train_loss: 0.018680026526657904
FOLD: 2, EPOCH: 19, valid_loss: 0.0162721311673522
FOLD: 2, EPOCH: 20, train_loss: 0.01826947122862955
F

FOLD: 5, EPOCH: 15, train_loss: 0.0197277173121162
FOLD: 5, EPOCH: 15, valid_loss: 0.016862811855971813
FOLD: 5, EPOCH: 16, train_loss: 0.019550081317116615
FOLD: 5, EPOCH: 16, valid_loss: 0.01682078529149294
FOLD: 5, EPOCH: 17, train_loss: 0.01930729465774533
FOLD: 5, EPOCH: 17, valid_loss: 0.016715194471180438
FOLD: 5, EPOCH: 18, train_loss: 0.01904511639252812
FOLD: 5, EPOCH: 18, valid_loss: 0.01665811099112034
FOLD: 5, EPOCH: 19, train_loss: 0.018685894353049144
FOLD: 5, EPOCH: 19, valid_loss: 0.016514243632555006
FOLD: 5, EPOCH: 20, train_loss: 0.01826834318158375
FOLD: 5, EPOCH: 20, valid_loss: 0.016329275630414485
FOLD: 5, EPOCH: 21, train_loss: 0.017794967381929865
FOLD: 5, EPOCH: 21, valid_loss: 0.016276067048311235
FOLD: 5, EPOCH: 22, train_loss: 0.01734247818259763
FOLD: 5, EPOCH: 22, valid_loss: 0.016231219619512557
FOLD: 5, EPOCH: 23, train_loss: 0.017006929659721802
FOLD: 5, EPOCH: 23, valid_loss: 0.0161650213599205
FOLD: 5, EPOCH: 24, train_loss: 0.01680143912430523
FOLD

...

FOLD: 6, EPOCH: 16, train_loss: 0.01968131048389438
FOLD: 6, EPOCH: 16, valid_loss: 0.01665901407599449
FOLD: 6, EPOCH: 17, train_loss: 0.01935443791503809
FOLD: 6, EPOCH: 17, valid_loss: 0.016557810492813588
FOLD: 6, EPOCH: 18, train_loss: 0.019054803262357
FOLD: 6, EPOCH: 18, valid_loss: 0.016544905975461006
FOLD: 6, EPOCH: 19, train_loss: 0.018730047646732556
FOLD: 6, EPOCH: 19, valid_loss: 0.016304928213357925
FOLD: 6, EPOCH: 20, train_loss: 0.01828881259374067
FOLD: 6, EPOCH: 20, valid_loss: 0.016269202791154385
FOLD: 6, EPOCH: 21, train_loss: 0.017835998343823312
FOLD: 6, EPOCH: 21, valid_loss: 0.016122772283852102
FOLD: 6, EPOCH: 22, train_loss: 0.01743214149788326
FOLD: 6, EPOCH: 22, valid_loss: 0.016048733107745648
FOLD: 6, EPOCH: 23, train_loss: 0.017090295162685468
FOLD: 6, EPOCH: 23, valid_loss: 0.016054935194551943
FOLD: 6, EPOCH: 24, train_loss: 0.016908668178025964
FOLD: 6, EPOCH: 24, valid_loss: 0.0160410663112998
FOLD: 0, EPOCH: 0, train_loss: 0.4683485254265216
FOLD: 

FOLD: 2, EPOCH: 19, valid_loss: 0.01630428120493889
FOLD: 2, EPOCH: 20, train_loss: 0.018327130714342707
FOLD: 2, EPOCH: 20, valid_loss: 0.01624854136258364
FOLD: 2, EPOCH: 21, train_loss: 0.01789336477970185
FOLD: 2, EPOCH: 21, valid_loss: 0.016123757287859915
FOLD: 2, EPOCH: 22, train_loss: 0.01745164067465432
FOLD: 2, EPOCH: 22, valid_loss: 0.016120854802429675
FOLD: 2, EPOCH: 23, train_loss: 0.01708070970564878
FOLD: 2, EPOCH: 23, valid_loss: 0.01609814140945673
FOLD: 2, EPOCH: 24, train_loss: 0.01692051316301028
FOLD: 2, EPOCH: 24, valid_loss: 0.01609212327748537
FOLD: 3, EPOCH: 0, train_loss: 0.46775013643006486
FOLD: 3, EPOCH: 0, valid_loss: 0.02506556048989296
FOLD: 3, EPOCH: 1, train_loss: 0.02397593917946021
FOLD: 3, EPOCH: 1, valid_loss: 0.019530980736017226
FOLD: 3, EPOCH: 2, train_loss: 0.021857527065642025
FOLD: 3, EPOCH: 2, valid_loss: 0.017723060846328735
FOLD: 3, EPOCH: 3, train_loss: 0.020694506525689244
FOLD: 3, EPOCH: 3, valid_loss: 0.017621143721044064
FOLD: 3, EPO

FOLD: 5, EPOCH: 23, train_loss: 0.017030298323402193
FOLD: 5, EPOCH: 23, valid_loss: 0.016205697283148766
FOLD: 5, EPOCH: 24, train_loss: 0.016840995169010293
FOLD: 5, EPOCH: 24, valid_loss: 0.016200288087129592
FOLD: 6, EPOCH: 0, train_loss: 0.4682961667374689
FOLD: 6, EPOCH: 0, valid_loss: 0.023935679346323013
FOLD: 6, EPOCH: 1, train_loss: 0.023703144077642433
FOLD: 6, EPOCH: 1, valid_loss: 0.01882742963731289
FOLD: 6, EPOCH: 2, train_loss: 0.02176343311308598
FOLD: 6, EPOCH: 2, valid_loss: 0.01863652914762497
FOLD: 6, EPOCH: 3, train_loss: 0.02064441244567738
FOLD: 6, EPOCH: 3, valid_loss: 0.01728231694549322
FOLD: 6, EPOCH: 4, train_loss: 0.020543536526106652
FOLD: 6, EPOCH: 4, valid_loss: 0.017736263647675513
FOLD: 6, EPOCH: 5, train_loss: 0.02035292519294486
FOLD: 6, EPOCH: 5, valid_loss: 0.017415036708116532
FOLD: 6, EPOCH: 6, train_loss: 0.020443188989547646
FOLD: 6, EPOCH: 6, valid_loss: 0.01747808124870062
FOLD: 6, EPOCH: 7, train_loss: 0.020396487812708024
FOLD: 6, EPOCH: 7

FOLD: 2, EPOCH: 2, train_loss: 0.021511870468048012
FOLD: 2, EPOCH: 2, valid_loss: 0.01824948236346245
FOLD: 2, EPOCH: 3, train_loss: 0.020637322800094577
FOLD: 2, EPOCH: 3, valid_loss: 0.01785211395472288
FOLD: 2, EPOCH: 4, train_loss: 0.0203211494550413
FOLD: 2, EPOCH: 4, valid_loss: 0.01730981882661581
FOLD: 2, EPOCH: 5, train_loss: 0.020325939734878184
FOLD: 2, EPOCH: 5, valid_loss: 0.017440279945731163
FOLD: 2, EPOCH: 6, train_loss: 0.020324301228028575
FOLD: 2, EPOCH: 6, valid_loss: 0.017741163186728956
FOLD: 2, EPOCH: 7, train_loss: 0.02036199134578105
FOLD: 2, EPOCH: 7, valid_loss: 0.017460601590573787
FOLD: 2, EPOCH: 8, train_loss: 0.020357054702582814
FOLD: 2, EPOCH: 8, valid_loss: 0.017867096588015555
FOLD: 2, EPOCH: 9, train_loss: 0.020345135040733278
FOLD: 2, EPOCH: 9, valid_loss: 0.017230405248701574
FOLD: 2, EPOCH: 10, train_loss: 0.02027962213622875
FOLD: 2, EPOCH: 10, valid_loss: 0.017497555129230024
FOLD: 2, EPOCH: 11, train_loss: 0.02024925870149314
FOLD: 2, EPOCH: 1

FOLD: 5, EPOCH: 5, valid_loss: 0.01745003677904606
FOLD: 5, EPOCH: 6, train_loss: 0.02033346342746498
FOLD: 5, EPOCH: 6, valid_loss: 0.01759981855750084
FOLD: 5, EPOCH: 7, train_loss: 0.020354990828402187
FOLD: 5, EPOCH: 7, valid_loss: 0.017423143833875655
FOLD: 5, EPOCH: 8, train_loss: 0.020354953233380706
FOLD: 5, EPOCH: 8, valid_loss: 0.017754039764404296
FOLD: 5, EPOCH: 9, train_loss: 0.02040766081994488
FOLD: 5, EPOCH: 9, valid_loss: 0.017494649216532706
FOLD: 5, EPOCH: 10, train_loss: 0.020353664409647993
FOLD: 5, EPOCH: 10, valid_loss: 0.01750331073999405
FOLD: 5, EPOCH: 11, train_loss: 0.020245835970656403
FOLD: 5, EPOCH: 11, valid_loss: 0.017169714607298374
FOLD: 5, EPOCH: 12, train_loss: 0.0201889721385273
FOLD: 5, EPOCH: 12, valid_loss: 0.01720014922320843
FOLD: 5, EPOCH: 13, train_loss: 0.020038961375854453
FOLD: 5, EPOCH: 13, valid_loss: 0.017180294953286646
FOLD: 5, EPOCH: 14, train_loss: 0.019912926036687123
FOLD: 5, EPOCH: 14, valid_loss: 0.016974197179079057
FOLD: 5, E

In [65]:
for i in range(len(target_cols)):
    fea=target_cols[i]
    test[fea]=predictions[:,i]

In [66]:
valid_results = train_targets_scored.drop(columns=target_cols).merge(train[['sig_id']+target_cols], on='sig_id', how='left').fillna(0)

y_true = train_targets_scored[target_cols].values
y_pred = valid_results[target_cols].values

score = 0
for i in range(len(target_cols)):
    score_ = log_loss(y_true[:, i], y_pred[:, i])
    score += score_ / target.shape[1]
    
print("CV log_loss: ", score)

CV log_loss:  0.014582831897729296


In [ ]:
# v2: 0.014681458126955248,0.01844
# v3: 0.014582831897729296,0.01839

In [ ]:
# tune parameters for pytorch 3
# tune dropout, hidden size.
#https://www.kaggle.com/kushal1506/moa-pytorch-optuna-hyperparameter-tuning